In [0]:
%run ../functions/functions

In [0]:
database_name = "dimensao"
table_name = "dm_socios"
target_path = f"{database_name}.{table_name}"
pk = "SK_SOCIOS"

In [0]:
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/SOCIOS_CONSOLIDADA/"
silver_path_q = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/QUALIFICACOES_CONSOLIDADA/"

In [0]:
df_s = spark.read.format("delta").load(silver_path_s)
df_q = spark.read.format("delta").load(silver_path_q)

In [0]:
df_s.createOrReplaceTempView("df_socios")
df_q.createOrReplaceTempView("df_qualificacao")

In [0]:
query = """SELECT 
  s.SK_SOCIOS,
  s.cnpj_basico as sk_cnpj_empresas_socios,
  s.identificador_socio,
  s.nome_socio,
  s.documento_cpf_cnpj,
  s.qualificacao_socio,
  q.descricao_qualificacao as qualificacao_socio_desc,
  s.data_entrada_sociedade,
  s.pais,
  s.cpf_representante_legal,
  s.nome_representante,
  s.qualificacao_representante_legal,
  c.descricao_qualificacao as qualificacao_representante_legal_desc,
  s.faixa_etaria,
  s.dt_ingestao
FROM df_socios as s 
  join df_qualificacao as q on s.qualificacao_socio = q.codigo_qualificacao
  join df_qualificacao as c on s.qualificacao_representante_legal = c.codigo_qualificacao;"""

In [0]:
df_join = spark.sql(query)

In [0]:
df_final = df_join.withColumn(
    "is_socio",
    F.when(
        F.lower(
            F.col("qualificacao_socio_desc")
        ).contains("socio"),
        1
    ).otherwise(0)
)

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS dimensao")

In [0]:
save_hive_table(df_final, target_path, pk)